# CustomGrid Environment Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/adversarial2dEnvAI/blob/master/notebooks/Environment_Demo.ipynb)

This notebook demonstrates how to use the `CustomGrid` environment and how the trained CNN is used for real-time item classification.

## 1. Setup

We need to install the package and some extra dependencies for headless rendering in Colab.

In [ ]:
!pip install git+https://github.com/dgaida/adversarial2dEnvAI.git
#!pip install pygame tensorflow numpy matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from custom_grid_env.interface import AgentInterface
from custom_grid_env.agents.random_player_agent import RandomPlayerAgent

# Set dummy video driver for headless environment
os.environ["SDL_VIDEODRIVER"] = "dummy"

## 2. Initialize the Environment

We use the `AgentInterface` to interact with the environment. Note that we set `render=True` but since we are in a headless environment, we will capture the frames manually.

In [ ]:
interface = AgentInterface(render=True, slip_probability=0.1)
agent = RandomPlayerAgent(interface.get_action_space())

obs = interface.reset()
img = interface.env.render() # Returns RGB array

plt.imshow(img)
plt.axis('off')
plt.show()

## 3. Running an Episode

We will run a few steps and visualize the state. If the agent lands on a dog or flower, the CNN (if loaded) will provide a prediction in the `info` dictionary.

In [ ]:
obs = interface.reset()
for i in range(10):
    action = agent.get_action(obs)
    obs, reward, done, info = interface.step(action)
    
    img = interface.env.render()
    
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Step {i+1}")
    plt.show()
    
    if "cnn_prediction" in info:
        label, prob = info["cnn_prediction"]
        print(f"--- CNN PREDICTION: {label} ({prob*100:.1f}%) ---")
    
    if done:
        print("Episode finished!")
        break

interface.close()

## 4. Understanding the Integration

The environment uses a pre-trained CNN model located at `src/custom_grid_env/cnn_tutorial/model.keras`. 

When `render()` is called, the `PygameRenderer` checks if the agent is on a cell with an item. If so, it:
1. Crops a 64x64 image of the cell.
2. Preprocesses the image (normalization).
3. Passes it through the CNN.
4. Adds the prediction to the `info` dictionary and displays it on the GUI info panel.